# Calgary 311 Service Requests: SQL Analysis
## Part 1: Get the data, check its quality, and clean it

**What this project is.** Calgary's 311 line lets residents report non-emergency issues like potholes, garbage carts, noise, and trees. The City publishes every request as open data. We will use SQL to answer real business questions about demand, service speed, and fairness across communities.

**What Part 1 does.**
1. Downloads 5 full years of requests (2021–2025) from the City's Open Data portal
2. Loads them into **DuckDB**, a database that runs inside this notebook
3. Checks data quality: missing values, duplicates, impossible dates
4. Builds a clean table that every later query will use

**How to run a notebook in Colab.** Click a grey code cell and press **Shift + Enter** to run it and move to the next one. Run the cells in order from top to bottom. A CPU runtime is fine, since you don't need a GPU for this.

**Why SQL inside Python?** Python only handles the download and displays results. **Every analysis step is written in SQL**, the skill we want to show employers. DuckDB uses standard SQL, so the same queries work in PostgreSQL, SQL Server, or Snowflake with little or no change.

## Step 1: Install and import the tools

In [ ]:
!pip -q install duckdb

import os, time, glob
import requests
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("DuckDB version:", duckdb.__version__)

## Step 2: Choose where to save the data

Colab deletes its files when the session ends. Saving to **Google Drive** means you download the data once and reuse it in Parts 2 and 3. When you run the next cell, Google will ask for permission to connect to your Drive. That's expected.

In [ ]:
SAVE_TO_DRIVE = True
try:
    if not SAVE_TO_DRIVE:
        raise RuntimeError("Drive saving turned off")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/calgary_311_project"
except Exception as e:
    print("Using local storage instead of Drive:", e)
    PROJECT_DIR = "calgary_311_project"

RAW_DIR = os.path.join(PROJECT_DIR, "raw_csv")
DB_PATH = os.path.join(PROJECT_DIR, "calgary_311.duckdb")
os.makedirs(RAW_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## Step 3: Download the data from Calgary Open Data

The City's portal has an **API**, a web address that returns data when you ask for it. The API has its own small query language, so we can ask for only what we need:

- `$select`: only the columns we'll use, which makes the download smaller
- `$where`: only requests made from 2021-01-01 to 2025-12-31, five complete years
- `$limit` and `$offset`: download in pages of 500,000 rows, since one giant file could fail halfway

We first ask the API **how many rows** match. That lets us confirm the download is complete afterwards.

This step takes about **10–20 minutes**. If the files are already in your Drive from an earlier run, it skips the download.

In [ ]:
BASE_URL = "https://data.calgary.ca/resource/iahh-g8bj.csv"
COLUMNS = ("service_request_id,requested_date,updated_date,closed_date,status_description,"
           "source,service_name,agency_responsible,comm_code,comm_name,longitude,latitude")
WHERE = "requested_date >= '2021-01-01T00:00:00' AND requested_date < '2026-01-01T00:00:00'"
PAGE_SIZE = 500_000
APP_TOKEN = ""   # optional: a free Socrata app token reduces the chance of being rate-limited

HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

def api_get(params, stream=False, tries=4):
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=600, stream=stream)
            r.raise_for_status()
            return r
        except Exception as e:
            print(f"  attempt {attempt} failed: {e}")
            time.sleep(10 * attempt)
    raise RuntimeError("Download failed after several attempts. Wait a few minutes and run this cell again.")

count_text = api_get({"$select": "count(*) AS n", "$where": WHERE}).text
EXPECTED_ROWS = int(count_text.strip().splitlines()[1].strip('"'))
print(f"The API reports {EXPECTED_ROWS:,} requests from 2021 to 2025.")

In [ ]:
existing = sorted(glob.glob(os.path.join(RAW_DIR, "part_*.csv")))
if existing:
    print(f"Found {len(existing)} files already downloaded, skipping. Delete the raw_csv folder to download again.")
else:
    offset, page = 0, 0
    while True:
        page += 1
        params = {"$select": COLUMNS, "$where": WHERE, "$order": ":id", "$limit": PAGE_SIZE, "$offset": offset}
        path = os.path.join(RAW_DIR, f"part_{page:02d}.csv")
        t0 = time.time()
        r = api_get(params, stream=True)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        rows = sum(1 for _ in open(path, encoding="utf-8")) - 1   # minus the header line
        print(f"Page {page}: {rows:,} rows in {time.time() - t0:.0f}s")
        if rows <= 0:
            os.remove(path)
        if rows < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    print("Download complete.")

## Step 4: Load the files into DuckDB

We load everything as **text** first (`all_varchar = true`). That may look odd, but it's a common, safe practice. If a date is malformed, loading it as text never fails, and we can find and count the bad values ourselves in Step 6. Loading as dates straight away could silently turn bad values into blanks.

`q()` is a small helper: it runs a SQL query and shows the result as a table.

In [ ]:
con = duckdb.connect(DB_PATH)

def q(sql):
    """Run a SQL query and return the result as a table."""
    return con.sql(sql).df()

con.sql(f"""
    CREATE OR REPLACE TABLE raw_requests AS
    SELECT *
    FROM read_csv('{RAW_DIR}/part_*.csv', header = true, all_varchar = true)
""")

loaded = q("SELECT COUNT(*) AS n FROM raw_requests")["n"][0]
print(f"Loaded {loaded:,} rows. The API said {EXPECTED_ROWS:,}.",
      "Complete." if loaded == EXPECTED_ROWS else "MISMATCH: the download may be incomplete. Delete raw_csv and rerun Step 3.")

## Step 5: First look at the data

Always look before analyzing. `DESCRIBE` lists the columns and their types. `LIMIT 5` shows a few real rows.

In [ ]:
q("DESCRIBE raw_requests")

In [ ]:
q("SELECT * FROM raw_requests LIMIT 5")

## Step 6: Data-quality checks

Before trusting any number, we check for common problems. Each query below answers one question. Read the comments inside the SQL (lines starting with `--`); they explain each part.

### 6a. How many values are missing in each column?
`COUNT(column)` counts only non-missing values, so `COUNT(*) - COUNT(column)` gives the number missing.

In [ ]:
q("""
    SELECT
        COUNT(*)                                            AS total_rows,
        COUNT(*) - COUNT(service_request_id)                AS missing_id,
        COUNT(*) - COUNT(requested_date)                    AS missing_requested_date,
        COUNT(*) - COUNT(closed_date)                       AS missing_closed_date,   -- expected for open requests
        COUNT(*) - COUNT(service_name)                      AS missing_service_name,
        COUNT(*) - COUNT(agency_responsible)                AS missing_agency,
        COUNT(*) - COUNT(comm_name)                         AS missing_community,
        COUNT(*) - COUNT(source)                            AS missing_source
    FROM raw_requests
""")

### 6b. Are there duplicate requests?
Each `service_request_id` should appear once. `GROUP BY ... HAVING COUNT(*) > 1` finds IDs that appear more than once.

In [ ]:
q("""
    WITH id_counts AS (                      -- a CTE: a named temporary result we can query next
        SELECT service_request_id, COUNT(*) AS times_seen
        FROM raw_requests
        GROUP BY service_request_id
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS duplicated_ids, COALESCE(SUM(times_seen), 0) AS rows_involved
    FROM id_counts
""")

### 6c. Do the dates convert properly?
`TRY_CAST` converts text to a timestamp and returns NULL instead of an error when it can't. A second format covers older export styles like `2018/10/17 12:00:00 AM`, just in case.

In [ ]:
con.sql("""
    CREATE OR REPLACE MACRO parse_ts(x) AS
        COALESCE(TRY_CAST(x AS TIMESTAMP), TRY_STRPTIME(x, '%Y/%m/%d %I:%M:%S %p'))
""")

q("""
    SELECT
        SUM(CASE WHEN requested_date IS NOT NULL AND parse_ts(requested_date) IS NULL THEN 1 ELSE 0 END) AS bad_requested_dates,
        SUM(CASE WHEN closed_date    IS NOT NULL AND parse_ts(closed_date)    IS NULL THEN 1 ELSE 0 END) AS bad_closed_dates,
        MIN(parse_ts(requested_date)) AS earliest_request,
        MAX(parse_ts(requested_date)) AS latest_request
    FROM raw_requests
""")

### 6d. Are there impossible dates, and are the times real?
A request can't be closed before it was opened. We also check how many timestamps are exactly midnight. If most are, the data only records the **day**, not the time, so resolution time must be measured in days.

In [ ]:
q("""
    SELECT
        SUM(CASE WHEN parse_ts(closed_date) < parse_ts(requested_date) THEN 1 ELSE 0 END)   AS closed_before_opened,
        ROUND(100.0 * AVG(CASE WHEN CAST(parse_ts(requested_date) AS TIME) = TIME '00:00:00'
                               THEN 1 ELSE 0 END), 1)                                        AS pct_requested_at_midnight,
        ROUND(100.0 * AVG(CASE WHEN CAST(parse_ts(closed_date) AS TIME) = TIME '00:00:00'
                               THEN 1 ELSE 0 END), 1)                                        AS pct_closed_at_midnight
    FROM raw_requests
""")

### 6e. What values do the category columns contain?
Checking the distinct values reveals typos, unexpected categories, and how requests are spread out.

In [ ]:
q("""
    SELECT status_description, COUNT(*) AS requests,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct   -- a window function: share of the grand total
    FROM raw_requests
    GROUP BY status_description
    ORDER BY requests DESC
""")

In [ ]:
q("""
    SELECT source AS channel, COUNT(*) AS requests,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM raw_requests
    GROUP BY source
    ORDER BY requests DESC
""")

In [ ]:
q("""
    SELECT
        COUNT(DISTINCT service_name)       AS service_types,
        COUNT(DISTINCT agency_responsible) AS agencies,
        COUNT(DISTINCT comm_name)          AS communities
    FROM raw_requests
""")

## Step 7: Build the clean table

Now we create the table every later query will use. The decisions, based on the checks above:

- **Dates** are converted to real timestamps.
- **Resolution time** is measured in whole days between request and closing. It's only calculated for requests that are closed and don't close before they open.
- **Duplicates**, if any, are removed by keeping one row per request ID.
- Useful helper columns are added: `request_year`, `request_month`, and `is_closed`.
- Rows with impossible dates are **kept but flagged** (`has_date_error`), so they're excluded from resolution-time analysis without distorting request counts.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE requests AS
    WITH typed AS (
        SELECT
            service_request_id,
            parse_ts(requested_date)          AS requested_at,
            parse_ts(updated_date)            AS updated_at,
            parse_ts(closed_date)             AS closed_at,
            TRIM(status_description)          AS status,
            TRIM(source)                      AS channel,
            TRIM(service_name)                AS service_name,
            TRIM(agency_responsible)          AS agency,
            UPPER(TRIM(comm_code))            AS comm_code,
            UPPER(TRIM(comm_name))            AS community,
            TRY_CAST(longitude AS DOUBLE)     AS longitude,
            TRY_CAST(latitude  AS DOUBLE)     AS latitude,
            ROW_NUMBER() OVER (PARTITION BY service_request_id
                               ORDER BY parse_ts(updated_date) DESC NULLS LAST) AS rn   -- 1 = most recent copy
        FROM raw_requests
    )
    SELECT
        * EXCLUDE (rn),
        YEAR(requested_at)                                  AS request_year,
        DATE_TRUNC('month', requested_at)                   AS request_month,
        closed_at IS NOT NULL                               AS is_closed,
        COALESCE(closed_at < requested_at, FALSE)           AS has_date_error,
        CASE WHEN closed_at >= requested_at
             THEN DATE_DIFF('day', CAST(requested_at AS DATE), CAST(closed_at AS DATE))
        END                                                 AS resolution_days
    FROM typed
    WHERE rn = 1
      AND requested_at IS NOT NULL
""")

q("DESCRIBE requests")

## Step 8: Checkpoint summary

This summary confirms the clean table looks right. **Copy the output of this cell and the checks in Step 6 back into the chat.** We'll use them to plan Part 2, the business questions.

In [ ]:
q("""
    SELECT
        request_year,
        COUNT(*)                                             AS requests,
        ROUND(100.0 * AVG(CAST(is_closed AS INT)), 1)        AS pct_closed,
        MEDIAN(resolution_days)                              AS median_days_to_close,
        SUM(CAST(has_date_error AS INT))                     AS date_errors
    FROM requests
    GROUP BY request_year
    ORDER BY request_year
""")

In [ ]:
con.close()
print("Saved database to", DB_PATH, "- Part 2 will open it directly, with no re-download needed.")